# References

https://sophwats.github.io/2018-04-05-gentle-als.html

https://stanford.edu/~rezab/classes/cme323/S15/notes/lec14.pdf

In [9]:
pip install numpy

  Using cached numpy-2.5.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached numpy-2.5.3-cp312-cp312-macosx_14_0_arm64.whl (5.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.3 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.0 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.
arviz 0.17.1 requires numpy<2.0,>=1.22.0, but you have numpy 2.5.3 which is incompatible.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but you have numpy 2.5.3 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.6 which is incompatible.
pytensor 2.23.0 requires numpy<2,>=1.17.0

In [10]:
import pandas as pd
import os
import numpy as np
from scipy.sparse import coo_matrix

In [11]:
print(os.getcwd())

/Users/prestonkou12/Desktop/School/TAMU/Fall2026/DAEN400/daen400_casestudies/03_Movie-Recommender


In [12]:
RATINGS = pd.read_csv("data/ratings.csv")
MOVIES = pd.read_csv("data/movies.csv")
TAGS = pd.read_csv("data/tags.csv")
LINKS = pd.read_csv("data/links.csv")

In [14]:
RATINGS[['userId', 'movieId']].nunique()

userId     200948
movieId     84432
dtype: int64

In [15]:
MOVIES['movieId'].nunique()

87585

In [13]:
display(RATINGS.head())

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [ ]:
TOTAL_GENRES = set.union(*MOVIES["genres"].str.split("|", regex=False).apply(set).dropna())

print(len(TOTAL_GENRES))

20


In [ ]:
display(RATINGS.head())

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [ ]:
display(LINKS.head())

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [ ]:
display(TAGS.head())

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [ ]:
user_ids = pd.Index(RATINGS["userId"].unique()).sort_values()
movie_ids = (
    pd.Index(MOVIES["movieId"].unique())
    .union(pd.Index(RATINGS["movieId"].unique()))
    .sort_values()
)

row = user_ids.get_indexer(RATINGS["userId"])
col = movie_ids.get_indexer(RATINGS["movieId"])

ratings_matrix = coo_matrix(
    (RATINGS["rating"].to_numpy(dtype=np.float32), (row, col)),
    shape=(len(user_ids), len(movie_ids)),
).tocsr()

RATINGS_WIDE = (
    pd.DataFrame.sparse.from_spmatrix(
        ratings_matrix, index=user_ids, columns=movie_ids
    )
    .astype(pd.SparseDtype("float32", 0))
)

display(RATINGS_WIDE.iloc[:5, :10])

,1,2,3,4,5,6,7,8,9,10
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,3.5,0,0,0,0,0,0,0,4.0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,4.0
